# DPD — Prueba individual por caso

Una celda por caso (1 a 18) de la hoja `Casos` del Excel. Cada celda:
1. Imprime título, descripción y resultado esperado del caso.
2. Filtra `scheduled_payments_installments` y `payment_tape` al contrato del caso.
3. Corre **ambos modos** (`cascade` y `join`) con `compute_from_data` (sin BD).
4. Muestra el detalle cuota-por-cuota con `dpd` y `arrears` en ambos modos.

Empezá corriendo la celda **setup** una sola vez. Después podés ejecutar cualquier `run_caso(N)` en orden o salteado.

Para casos cascade-específicos podés llamar `run_caso(N, partial_counts=True)` o cambiar `calc_date=date(...)`.

In [ ]:
# === Setup — correr una sola vez ===
import os
import sys
from datetime import date
from decimal import Decimal

import pandas as pd

sys.path.insert(0, os.path.abspath("."))
from dpd.config import RunConfig
from dpd.modes import join_installment, cascade_fifo

EXCEL_PATH = "tests/Days Past Due.xlsx"
CALC_DATE = date(2026, 10, 3)

# === Resultados esperados por caso ===
# Parseados a mano desde la columna "Resultado esperado del script" (prosa en español).
# `dpd` y `arrears` son los valores AGREGADOS POR CONTRATO (max DPD entre cuotas, suma de arrears).
# `mode_pref` indica qué modo se considera "el correcto" para ese caso ("cascade" | "join" | "ambos").
# `note` aclara cualquier ambigüedad o por qué los modos difieren.
#
# Referencia de fechas vs corte 2026-10-03:
#   jun-01 → 124 días | jul-01 → 94 | ago-01 → 63 | sep-01 → 32 | oct-01 → 2
EXPECTED = {
    1:  {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Happy path: pagos exactos a las 5 cuotas vigentes."},
    2:  {"dpd": 2,   "arrears": 132253,    "mode_pref": "ambos",
         "note": "Mitad de oct sin pagar; oct-1 a oct-3 = 2 días."},
    3:  {"dpd": 124, "arrears": None,      "mode_pref": "ambos",
         "note": "DPD desde primera cuota impaga (jun-01). Arrears = suma cuotas vencidas + moratorios (no calculamos moratorios)."},
    4:  {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Pagó tarde pero ya está al día al 3-oct. Histórico=15 días NO se ve en dpd_current."},
    5:  {"dpd": 63,  "arrears": 132253,    "mode_pref": "join",
         "note": "Prose dice 'DPD desde vencimiento de ago' → join=63. Cascade redistribuye y la mora cae a oct (=2)."},
    6:  {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Pagos anticipados; al 3-oct todo cubierto."},
    7:  {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Sobrepago en oct; excedente queda como saldo a favor (no se mueve a cuota futura en este código)."},
    8:  {"dpd": 63,  "arrears": 396759,    "mode_pref": "join",
         "note": "Prose dice DPD desde ago (la más vieja impaga). Cascade redistribuye → dpd_max cae a sep (=32)."},
    9:  {"dpd": 0,   "arrears": 132253,    "mode_pref": "cascade+partial",
         "note": "Excedente de sep cubre mitad de oct. Requiere partial_counts=True para que cascade marque oct como pagada."},
    10: {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Dos pagos parciales en oct suman la cuota completa."},
    11: {"dpd": 94,  "arrears": None,      "mode_pref": "ambos",
         "note": "Pago de sep (con ref=jun) cubre solo jun. La más vieja impaga es jul → DPD=94."},
    12: {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Pago el día del corte cubre la cuota; (calc_date - installment_date) = 0 días."},
    13: {"dpd": None,"arrears": None,      "mode_pref": "?",
         "note": "Tolerancia de redondeo NO definida en el código actual: 0.50 de menos lo deja como impago."},
    14: {"dpd": 0,   "arrears": 0,         "mode_pref": "?",
         "note": "Requiere que el Excel tenga el schedule REESTRUCTURADO. Hoy hay 10 cuotas planas → no se puede testear."},
    15: {"dpd": 0,   "arrears": 0,         "mode_pref": "ambos",
         "note": "Pago duplicado en ago se considera saldo a favor; el resto pagado correctamente."},
    16: {"dpd": 124, "arrears": 264506,    "mode_pref": "join",
         "note": "Jun nunca pagó. join=124 (correcto). Cascade redistribuye y reporta dpd=2 sobre oct."},
    17: {"dpd": 124, "arrears": None,      "mode_pref": "ambos",
         "note": "Sin payment tape; DPD desde la primera cuota."},
    18: {"dpd": None,"arrears": None,      "mode_pref": "?",
         "note": "Depende de cuándo se abandonó. Sin estructurar en la prose; revisar el resultado contra el Excel."},
}

# Cargar las 3 hojas
xls = pd.ExcelFile(EXCEL_PATH)
cases_df = pd.read_excel(xls, sheet_name="Casos")
sched_df = pd.read_excel(xls, sheet_name="scheduled_payments_installments")
pay_df = pd.read_excel(xls, sheet_name="Payment_Tape")

# Normalizar tipos
sched_df["date"] = pd.to_datetime(sched_df["date"]).dt.date
pay_df["payment_date"] = pd.to_datetime(pay_df["payment_date"]).dt.date

def _ref_to_str(s):
    if pd.api.types.is_float_dtype(s):
        return s.where(s.isna(), s.astype("Int64")).astype(str)
    return s.astype(str)

sched_df["borrower_installment_reference"] = _ref_to_str(sched_df["borrower_installment_reference"])
pay_df["borrower_installment_reference"] = _ref_to_str(pay_df["borrower_installment_reference"])

sched_df = sched_df.reset_index(drop=True)
sched_df["id"] = sched_df.index + 1


def installments_from_df(df):
    return [
        {
            "id": int(r["id"]),
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r["borrower_installment_reference"],
            "installment_date": r["date"],
            "gross_amount": r.get("gross_amount", 0),
            "guarantee_amount": r.get("guarantee_amount", 0),
            "principal_amount": r.get("principal_amount", 0),
            "interest_amount": r.get("interest_amount", 0),
            "tax_amount": r.get("tax_amount", 0),
            "fee_amount": r.get("fee_amount", 0),
        }
        for _, r in df.iterrows()
    ]


def payments_from_df(df):
    return [
        {
            "borrower_contract_id": r["borrower_contract_id"],
            "borrower_installment_reference": r.get("borrower_installment_reference"),
            "payment_date": r["payment_date"],
            "total_payment": r.get("total_payment", 0),
        }
        for _, r in df.iterrows()
    ]


def _check(actual, expected):
    if expected is None:
        return "?"
    return "✓" if int(actual) == int(expected) else "✗"


def run_caso(n, partial_counts=False, calc_date=None):
    """Corre los modos cascade + join para el contrato del caso N y muestra el detalle."""
    if calc_date is None:
        calc_date = CALC_DATE
    case_row = cases_df[cases_df["#"] == n]
    if case_row.empty:
        print(f"⚠ Caso {n} no encontrado en la hoja Casos.")
        return None
    case = case_row.iloc[0]
    contract_id = case["Borrower contract id"]
    exp = EXPECTED.get(n, {"dpd": None, "arrears": None, "mode_pref": "?", "note": ""})

    print(f"{'═' * 88}")
    print(f"  Caso {n}: {case['Caso']}")
    print(f"{'═' * 88}")
    print(f"  Contrato: {contract_id}")
    print(f"  Fecha de corte: {calc_date}  |  partial_counts: {partial_counts}")
    print()
    print(f"  Descripción:")
    print(f"    {case['Descripción']}")
    print()
    print(f"  Esperado (prose del Excel):")
    print(f"    {case['Resultado esperado del script']}")
    print()

    sched_sub = sched_df[sched_df["borrower_contract_id"] == contract_id].copy()
    pay_sub = pay_df[pay_df["borrower_contract_id"] == contract_id].copy()

    insts = installments_from_df(sched_sub)
    pays = payments_from_df(pay_sub)

    cfg_c = RunConfig(company_id=1, company_code="*", mode="cascade",
                      partial_payment_counts=partial_counts, calculation_date=calc_date)
    cfg_j = RunConfig(company_id=1, company_code="*", mode="join",
                      partial_payment_counts=False, calculation_date=calc_date)

    c_results = list(cascade_fifo.compute_from_data(insts, pays, cfg_c))
    j_results = list(join_installment.compute_from_data(insts, pays, cfg_j))

    cdf = pd.DataFrame(c_results).rename(
        columns={"dpd_current": "dpd_cascade", "amount_in_arrears": "arrears_cascade"}
    )
    jdf = pd.DataFrame(j_results).rename(
        columns={"dpd_current": "dpd_join", "amount_in_arrears": "arrears_join"}
    )

    detail = (
        sched_sub[["id", "borrower_installment_reference", "date", "gross_amount",
                   "principal_amount", "interest_amount"]]
        .merge(cdf, on="id", how="left")
        .merge(jdf, on="id", how="left")
        .sort_values("date")
    )

    if not pay_sub.empty:
        paid_by_ref = (
            pay_sub.groupby("borrower_installment_reference", as_index=False)
                   .agg(total_paid_by_ref=("total_payment", "sum"),
                        n_pagos=("total_payment", "count"),
                        last_payment=("payment_date", "max"))
        )
        detail = detail.merge(paid_by_ref, on="borrower_installment_reference", how="left")
    else:
        detail["total_paid_by_ref"] = 0
        detail["n_pagos"] = 0
        detail["last_payment"] = pd.NaT

    detail["total_paid_by_ref"] = detail["total_paid_by_ref"].fillna(0)
    detail["n_pagos"] = detail["n_pagos"].fillna(0).astype(int)

    dpd_c_max = max((r["dpd_current"] for r in c_results), default=0)
    arr_c_sum = sum((r["amount_in_arrears"] for r in c_results), Decimal(0))
    dpd_j_max = max((r["dpd_current"] for r in j_results), default=0)
    arr_j_sum = sum((r["amount_in_arrears"] for r in j_results), Decimal(0))
    n_cuotas = len(detail)
    n_pagos_total = len(pay_sub)
    total_pagado = pay_sub["total_payment"].sum() if not pay_sub.empty else 0

    print(f"  Datos del contrato: {n_cuotas} cuotas | {n_pagos_total} pagos | total pagado: {total_pagado:,.0f}")
    print()
    print(f"  ┌─────────────────────────────────────────────────────────────────────────────┐")
    print(f"  │ DÍAS DE MORA (DPD): comparación esperado vs calculado                       │")
    print(f"  ├─────────────────────────────────────────────────────────────────────────────┤")
    exp_dpd = exp["dpd"]
    exp_dpd_str = "?" if exp_dpd is None else f"{exp_dpd}"
    print(f"  │   Esperado:                          dpd = {exp_dpd_str:<6}                              │")
    print(f"  │   Calculado CASCADE:                 dpd = {dpd_c_max:<6}    {_check(dpd_c_max, exp_dpd)}                       │")
    print(f"  │   Calculado JOIN:                    dpd = {dpd_j_max:<6}    {_check(dpd_j_max, exp_dpd)}                       │")
    print(f"  └─────────────────────────────────────────────────────────────────────────────┘")
    print()
    print(f"  Modo recomendado por el caso: {exp['mode_pref']}")
    print(f"  Nota: {exp['note']}")
    print()
    print(f"  Arrears (informativo, no comparado):")
    print(f"    CASCADE: {arr_c_sum:,.2f}  (esperado: {exp['arrears'] if exp['arrears'] is not None else '?'})")
    print(f"    JOIN:    {arr_j_sum:,.2f}")

    return detail


print(f"Setup OK — {len(cases_df)} casos cargados, {len(sched_df)} cuotas, {len(pay_df)} pagos")

In [ ]:
run_caso(1)

In [ ]:
run_caso(2)

In [ ]:
run_caso(3)

In [ ]:
run_caso(4)

In [ ]:
run_caso(5)

In [ ]:
run_caso(6)

In [ ]:
run_caso(7)

In [ ]:
run_caso(8)

In [ ]:
run_caso(9)

In [ ]:
run_caso(10)

In [ ]:
run_caso(11)

In [ ]:
run_caso(12)

In [ ]:
run_caso(13)

In [ ]:
run_caso(14)

In [ ]:
run_caso(15)

In [ ]:
run_caso(16)

In [ ]:
run_caso(17)

In [ ]:
run_caso(18)